# 2 - Standalone checkpoint inference

In [ ]:
import digitalhub as dh

In [ ]:
project = dh.get_or_create_project("cflv-1")

In [ ]:
artifact_names = {
    "training_results": "2-abmil-conch-v15-results",
    "train_features": "1-trident-features-train",
    "val_features": "1-trident-features-val",
    "internal_test_features": "1-trident-features-internal-test",
    "external_test_features": "1-trident-features-external-test",
}
entity_ids = {}
inputs = {key: project.get_artifact(name, **({"entity_id": entity_ids[key]} if key in entity_ids else {}))
          for key, name in artifact_names.items()}

## Declaration

In [ ]:
inference_function = project.new_function(
    name="2-checkpoint-inference", kind="python", python_version="PYTHON3_11",
    requirements="2-inference/requirements.txt", code_src="2-inference",
    handler="main:predict_classifier",
)
print(inference_function.key)

In [ ]:
inference_function = project.get_function("2-checkpoint-inference",
    entity_id=""
)

## Building

In [ ]:
inference_build = inference_function.run(action="build",
    instructions=["apt-get update && apt-get install -y git libopenslide0 libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*"],
    resources={"cpu": "8", "mem": "16Gi", "disk": "20Gi"},
)
print(inference_build.key)

In [ ]:
image_ref = inference_function.refresh().spec.image

## Running

In [ ]:
dh.get_k8s_resource_profiles()

In [ ]:
inference_run = inference_function.run(
    action="job", inputs={name: artifact.key for name, artifact in inputs.items()},
    parameters={
        "output_artifact_name": "2-inference-results",
        "run_name": None,  # Required if the training artifact contains multiple runs.
        "checkpoint_name": "best_model.pt",
        "batch_size": 32, "num_workers": 4, "save_train": False,
    },
    profile="1xl40s",
    resources={"cpu": "8", "mem": "32Gi", "gpu": "1", "disk": "20Gi"},
    local_execution=False,
)
print(inference_run.key)